# AI-Powered Lung Disease Detection
## Preprocessing, Exploratory Data Analysis, and YOLOX Training

In [ ]:
%%capture
# Install PyTorch with CUDA
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install additional dependencies
!pip install matplotlib pandas pillow torchtnt==0.2.0 tqdm opencv-python seaborn

# Install extra dependencies for pandas
!pip install tabulate pyarrow fastparquet

# Install package for creating visually distinct colormaps
!pip install distinctipy

# Install utility packages
!pip install cjm_pandas_utils cjm_psl_utils cjm_pil_utils cjm_pytorch_utils cjm_yolox_pytorch cjm_torchvision_tfms

In [ ]:
# Import Python Standard Library dependencies
import datetime
from functools import partial
from glob import glob
import json
import math
import multiprocessing
import os
from pathlib import Path
import random

# Import utility functions
import cjm_pil_utils
from cjm_psl_utils.core import download_file, file_extract
from cjm_pil_utils.core import resize_img, get_img_files, stack_imgs
from cjm_pytorch_utils.core import tensor_to_pil, get_torch_device, set_seed, denorm_img_tensor
from cjm_pandas_utils.core import markdown_to_pandas, convert_to_numeric, convert_to_string
from cjm_torchvision_tfms.core import ResizeMax, PadSquare, CustomRandomIoUCrop, CustomRandomAugment

# Import YOLOX package
from cjm_yolox_pytorch.model import build_model, MODEL_CFGS, NORM_STATS
from cjm_yolox_pytorch.utils import generate_output_grids
from cjm_yolox_pytorch.loss import YOLOXLoss
from cjm_yolox_pytorch.inference import YOLOXInferenceWrapper

# Import the distinctipy module
from distinctipy import distinctipy

# Import matplotlib and seaborn for creating plots
import matplotlib.pyplot as plt
import seaborn as sns

# Import numpy and cv2
import numpy as np
import cv2

# Import the pandas package
import pandas as pd

# Do not truncate the contents of cells and display all rows and columns
pd.set_option('max_colwidth', None, 'display.max_rows', None, 'display.max_columns', None)

# Import PIL for image manipulation
from PIL import Image

# Import PyTorch dependencies
import torch
from torch.amp import autocast
from torch.cuda.amp import GradScaler
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchtnt.utils import get_module_summary

# Import torchvision dependencies
import torchvision
torchvision.disable_beta_transforms_warning()
from torchvision.tv_tensors import BoundingBoxes
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms.v2  as transforms
from torchvision.transforms.v2 import functional as TF

# Import tqdm for progress bar
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

## Dataset Setup and Initialization
We will set random seeds for reproducibility and set up the dataset paths.

In [ ]:
seed = 42
set_seed(seed)
device = get_torch_device() # Returns string like 'cuda' or 'cpu'
dtype = torch.float32

train_sz = 256
bs = 16
epochs = 50
lr = 1e-3

project_dir = Path("/kaggle/working/yolox_lung_tumor")
project_dir.mkdir(parents=True, exist_ok=True)

image_dir = "/kaggle/input/pidata-new-names/Dataset/Images"
mask_dir = "/kaggle/input/pidata-new-names/Dataset/Annotations"

if not os.path.exists(image_dir):
    print("Dataset directories not found. Please ensure the dataset is added to the Kaggle notebook.")

## Preprocessing: Mask to Bounding Box Extraction
YOLOX requires bounding box coordinates rather than segmentation masks. Here we find the contours of the active regions in the mask and create encompassing bounding boxes.

In [ ]:
def mask_to_bbox(mask_path):
    """Convert segmentation mask to bounding box"""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    
    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    bboxes = []
    for contour in contours:
        if cv2.contourArea(contour) > 50:  # Filter small noise
            x, y, w, h = cv2.boundingRect(contour)
            bboxes.append([x, y, x+w, y+h])  # xmin, ymin, xmax, ymax
            
    return bboxes

image_files = [f for f in os.listdir(image_dir) if f.endswith('.png')] if os.path.exists(image_dir) else []
mask_files = [f for f in os.listdir(mask_dir) if f.endswith('.png')] if os.path.exists(mask_dir) else []

common_files = list(set([os.path.splitext(f)[0] for f in image_files]) & set([os.path.splitext(f)[0] for f in mask_files]))

data = []
if len(common_files) > 0:
    print("Extracting bounding boxes from masks...")
    for f in tqdm(common_files):
        img_name = f + '.png'
        mask_path = os.path.join(mask_dir, img_name)
        bboxes = mask_to_bbox(mask_path)
        
        for bbox in bboxes:
            data.append({
                'image': img_name,
                'xmin': bbox[0],
                'ymin': bbox[1],
                'xmax': bbox[2],
                'ymax': bbox[3],
                'width': bbox[2] - bbox[0],
                'height': bbox[3] - bbox[1],
                'area': (bbox[2] - bbox[0]) * (bbox[3] - bbox[1]),
                'label': 'tumor'
            })

df = pd.DataFrame(data)
print(f"Total valid images with masks: {len(common_files)}")
print(f"Total bounding box annotations extracted: {len(df)}")
if len(df) > 0:
    display(df.head())

## Exploratory Data Analysis (EDA)
Let's visualize the transformation from the raw image and mask to the object detection bounding box.

In [ ]:
if len(df) > 0:
    # Look at a few random samples
    samples = df['image'].sample(3, random_state=seed).unique()
    
    fig, axes = plt.subplots(len(samples), 3, figsize=(15, 5 * len(samples)))
    if len(samples) == 1: axes = [axes]
    
    for i, img_name in enumerate(samples):
        img_path = os.path.join(image_dir, img_name)
        mask_path = os.path.join(mask_dir, img_name)
        
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Image with BBoxes
        img_bbox = img.copy()
        img_rows = df[df['image'] == img_name]
        for _, row in img_rows.iterrows():
            cv2.rectangle(img_bbox, (int(row['xmin']), int(row['ymin'])), (int(row['xmax']), int(row['ymax'])), (255, 0, 0), 3)
            
        axes[i][0].imshow(img)
        axes[i][0].set_title(f"Original Image: {img_name}")
        axes[i][0].axis('off')
        
        axes[i][1].imshow(mask, cmap='gray')
        axes[i][1].set_title("Segmentation Mask")
        axes[i][1].axis('off')
        
        axes[i][2].imshow(img_bbox)
        axes[i][2].set_title("Extracted Bounding Boxes")
        axes[i][2].axis('off')
        
    plt.tight_layout()
    plt.show()

### Bounding Box Statistics
Understanding the size distribution of the bounding boxes is crucial for anchor-free object detectors like YOLOX to ensure there's enough resolution to detect the objects.

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sns.histplot(df['width'], bins=30, ax=axes[0], color='skyblue', kde=True)
    axes[0].set_title('Distribution of BBox Widths')
    
    sns.histplot(df['height'], bins=30, ax=axes[1], color='salmon', kde=True)
    axes[1].set_title('Distribution of BBox Heights')
    
    sns.histplot(df['area'], bins=30, ax=axes[2], color='lightgreen', kde=True)
    axes[2].set_title('Distribution of BBox Areas')
    
    plt.tight_layout()
    plt.show()
    
    print("Bounding Box Summary Statistics:")
    display(df[['width', 'height', 'area']].describe())

## Dataset Splitting
We randomly split the dataset into 80% Training, 10% Validation, and 10% Testing sets.

In [ ]:
if len(df) > 0:
    all_images = df['image'].unique().tolist()
    random.shuffle(all_images)

    train_split = int(0.8 * len(all_images))
    val_split = int(0.9 * len(all_images))

    train_keys = all_images[:train_split]
    val_keys = all_images[train_split:val_split]
    test_keys = all_images[val_split:]

    class_names = sorted(df['label'].unique().tolist())
    class_to_idx = {name: i for i, name in enumerate(class_names)}

    print(f"Classes: {class_names}")
    print(f"Train Images: {len(train_keys)}, Val Images: {len(val_keys)}, Test Images: {len(test_keys)}")
else:
    train_keys, val_keys, test_keys, class_names, class_to_idx = [], [], [], [], {}

## Custom PyTorch Dataset
Defining the `LungTumorDataset` capable of decoding our dataframe values into Torchvision `BoundingBoxes`.

In [ ]:
class LungTumorDataset(Dataset):
    def __init__(self, img_keys, df, images_path, class_to_idx, transforms=None):
        self.img_keys = img_keys
        self.df = df.set_index('image')
        self.images_path = Path(images_path)
        self.class_to_idx = class_to_idx
        self.transforms = transforms

    def __len__(self):
        return len(self.img_keys)

    def __getitem__(self, idx):
        img_name = self.img_keys[idx]
        img_path = self.images_path/img_name
        image = Image.open(img_path).convert("RGB")
        
        rows = self.df.loc[[img_name]]
        bboxes = rows[['xmin', 'ymin', 'xmax', 'ymax']].values
        labels = [self.class_to_idx[l] for l in rows['label'].values]
        
        bboxes = torch.tensor(bboxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.long)
        
        target = {
            'boxes': BoundingBoxes(bboxes, format="xyxy", canvas_size=image.size[::-1]),
            'labels': labels
        }
        
        if self.transforms:
            image, target = self.transforms(image, target)
            
        return image, target

## Data Augmentation and Transforms
We use `torchvision.transforms.v2` along with custom augmentations suitable for modern object detection.

In [ ]:
fill = (0, 0, 0)
norm_stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

resize_pad_tfm = transforms.Compose([
    ResizeMax(max_sz=train_sz),
    PadSquare(shift=True, fill=fill),
    transforms.Resize([train_sz] * 2, antialias=True)
])

final_tfms = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.SanitizeBoundingBoxes(),
    transforms.Normalize(*norm_stats),
])

train_tfms = transforms.Compose([
    CustomRandomAugment(),
    resize_pad_tfm,
    final_tfms,
])

valid_tfms = transforms.Compose([resize_pad_tfm, final_tfms])

if len(df) > 0:
    train_dataset = LungTumorDataset(train_keys, df, image_dir, class_to_idx, train_tfms)
    valid_dataset = LungTumorDataset(val_keys, df, image_dir, class_to_idx, valid_tfms)

    num_workers = 0 
    data_loader_params = {
        'batch_size': bs,
        'num_workers': num_workers,
        'collate_fn': lambda batch: tuple(zip(*batch)),
        'pin_memory': 'cuda' in device,
        'pin_memory_device': device if 'cuda' in device else ''
    }

    train_loader = DataLoader(train_dataset, **data_loader_params, shuffle=True, drop_last=True)
    valid_loader = DataLoader(valid_dataset, **data_loader_params, drop_last=False)

    print(f"Train batches: {len(train_loader)}, Valid batches: {len(valid_loader)}")

### Visualizing the Data Loader
Checking our augmentation pipeline and bounds mapping before passing data to the model.

In [ ]:
if len(df) > 0:
    # Fetch a single batch
    for images, targets in train_loader:
        break
        
    fig, axes = plt.subplots(2, min(4, bs), figsize=(20, 10))
    axes = axes.flatten()
    
    colors = distinctipy.get_colors(len(class_names))
    int_colors = [tuple(int(c*255) for c in color) for color in colors]
    
    for i in range(min(8, bs)):
        img_tensor = denorm_img_tensor(images[i], norm_stats)
        img_uint8 = transforms.ToDtype(torch.uint8, scale=True)(img_tensor)
        
        boxes = targets[i]['boxes']
        labels = [class_names[int(l)] for l in targets[i]['labels']]
        
        annotated = draw_bounding_boxes(
            image=img_uint8,
            boxes=boxes,
            labels=labels,
            colors=[int_colors[class_names.index(l)] for l in labels],
            width=2
        )
        
        axes[i].imshow(tensor_to_pil(annotated))
        axes[i].axis('off')
        
    plt.tight_layout()
    plt.show()

## Model Tracking & Loss Function
We initialize the `yolox_tiny` model. Note that we define the loss function `YOLOXLoss` with a custom bounding box loss weight.

In [ ]:
if len(df) > 0:
    model_type = 'yolox_tiny'
    model = build_model(model_type, len(class_names), pretrained=True).to(device)

    loss_func = YOLOXLoss(num_classes=len(class_names), bbox_loss_weight=10.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * len(train_loader))

## Training Loop
Custom training loop utilizing PyTorch's `autocast` for Automatic Mixed Precision (AMP), speeding up training significantly while reducing memory footprint.

In [ ]:
def run_epoch(model, loader, optimizer, scheduler, loss_func, device, scaler, is_train):
    model.train() if is_train else model.eval()
    total_loss = 0
    
    if len(loader) == 0:
        return 0.0
        
    pbar = tqdm(loader, desc="Train" if is_train else "Eval")
    
    for batch_id, (inputs, targets) in enumerate(pbar):
        inputs = torch.stack(inputs).to(device)
        gt_boxes = [t['boxes'].to(device) for t in targets]
        gt_labels = [t['labels'].to(device) for t in targets]
        
        # Forward pass with Automatic Mixed Precision (AMP)
        with autocast(device_type=torch.device(device).type):
            cls_scores, bbox_preds, obj_scores = model(inputs)
            losses = loss_func(cls_scores, bbox_preds, obj_scores, gt_boxes, gt_labels)
            loss = sum(losses.values())
            
        if is_train:
            optimizer.zero_grad()
            if scaler:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            
            scheduler.step()
            
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item(), avg_loss=total_loss/(batch_id+1))
        
        # Safety check for NaN loss
        if is_train and not math.isfinite(loss.item()):
            print(f"Loss is {loss.item()}, stopping training")
            break
        
    return total_loss / len(loader)

if len(df) > 0:
    scaler = GradScaler() if 'cuda' in device else None
    best_val_loss = float('inf')

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        train_loss = run_epoch(model, train_loader, optimizer, scheduler, loss_func, device, scaler, True)
        with torch.no_grad():
            val_loss = run_epoch(model, valid_loader, None, None, loss_func, device, scaler, False)
            
        if val_loss < best_val_loss and val_loss > 0:
            best_val_loss = val_loss
            torch.save(model.state_dict(), project_dir/"best_model.pth")
            print("Saved best model!")

## Inference
Validating the trained model locally. We wrap the model in `YOLOXInferenceWrapper` and apply real-time tensor transformations.

In [ ]:
if len(df) > 0 and (project_dir/"best_model.pth").exists():
    model.load_state_dict(torch.load(project_dir/"best_model.pth"))
    model.eval()

    mean_t = torch.tensor(norm_stats[0]).view(1, 3, 1, 1).to(device)
    std_t = torch.tensor(norm_stats[1]).view(1, 3, 1, 1).to(device)
    wrapped_model = YOLOXInferenceWrapper(model, mean_t, std_t)

    def predict(img_path):
        img = Image.open(img_path).convert("RGB")
        resized = resize_img(img, target_sz=train_sz)
        input_t = transforms.Compose([transforms.ToImage(), transforms.ToDtype(torch.float32, scale=True)])(resized)[None].to(device)
        
        with torch.no_grad():
            output = wrapped_model(input_t).cpu()
        
        conf_thresh = 0.3
        mask = output[0, :, -1] > conf_thresh
        proposals = output[0, mask]
        
        if len(proposals) == 0: return resized
        
        boxes = torchvision.ops.box_convert(proposals[:, :4], 'xywh', 'xyxy')
        labels = [class_names[int(i)] for i in proposals[:, 4]]
        
        colors = distinctipy.get_colors(len(class_names))
        int_colors = [tuple(int(c*255) for c in color) for color in colors]
        
        annotated = draw_bounding_boxes(
            image=(transforms.PILToTensor()(resized)),
            boxes=boxes,
            labels=labels,
            colors=[int_colors[class_names.index(l)] for l in labels],
            width=2
        )
        return tensor_to_pil(annotated)

### Visualization of Test Set Predictions
We plot original images vs their predicted outputs to get an intuitive grasp of the model's accuracy.

In [ ]:
if len(df) > 0 and (project_dir/"best_model.pth").exists():
    print(f"Best Validation Loss: {best_val_loss:.4f}")

    # Visualize predictions on a few random test images
    num_test = min(4, len(test_keys))
    if num_test > 0:
        fig, axes = plt.subplots(num_test, 2, figsize=(12, 5 * num_test))
        if num_test == 1: axes = [axes]
        images_path = Path(image_dir)
        
        for i in range(num_test):
            test_img_name = random.choice(test_keys)
            test_img_path = images_path/test_img_name
            
            # Show original
            orig_img = Image.open(test_img_path).convert("RGB")
            axes[i][0].imshow(orig_img)
            axes[i][0].set_title(f"Original: {test_img_name}")
            axes[i][0].axis('off')
            
            # Show prediction
            prediction = predict(test_img_path)
            axes[i][1].imshow(prediction)
            axes[i][1].set_title(f"Prediction: {test_img_name}")
            axes[i][1].axis('off')
            
        plt.tight_layout()
        plt.show()
    else:
        print("No test images available or image directory not found.")